# KXBTC15M Bitcoin 15m Up/Down Due Diligence

## 1. Title / Purpose

This notebook is a human-readable market-structure due-diligence workbench for Kalshi `KXBTC15M` Bitcoin 15-minute up/down contracts. It helps a researcher inspect contract mechanics, data quality, segmentation, dumb baselines, calibration, PnL attribution, and exact-window persistence in one lightweight pass.

It is not a trading bot, model-training notebook, live-deployment gate, database writer, runtime change, or agent orchestration surface. Any live-money decision needs separate evidence, controls, and approval outside this notebook.


## 2. Parameters

These assumptions are intentionally visible at the top. The notebook buys one contract per qualifying row, values entry at the selected side ask, subtracts the fee and slippage assumptions below, and uses stressed slippage only for a conservative ROI read.

`REFRESH_SOURCE_DATA = False` means exact-window runs use cached raw public source artifacts when present and refresh only when the required cache is missing. If a required current / 7d / 30d window cannot be loaded completely, execution stops.


In [1]:
from __future__ import annotations

import json
import time
from collections import Counter
from datetime import UTC, date, datetime, time as datetime_time, timedelta
from pathlib import Path
from typing import Any
from urllib import parse, request

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = NOTEBOOK_CWD.parents[1] if NOTEBOOK_CWD.name == "notebooks" else NOTEBOOK_CWD

market_series = "KXBTC15M"
date_start = date(2026, 6, 5)
date_end = date(2026, 6, 5)
source_mode = "kalshi_public"
data_path = None
output_dir = REPO_ROOT / "artifacts/market-due-diligence/kxbtc15m-public-20260605"
source_cache_root = REPO_ROOT / "artifacts/market-due-diligence/kxbtc15m-public-source-cache"
REFRESH_SOURCE_DATA = False

window_specs = (
    {"window": "current", "days": 1},
    {"window": "7d", "days": 7},
    {"window": "30d", "days": 30},
)
primary_policy = "expensive_yes"

price_buckets = (
    {"label": "cheap", "min_price": None, "max_price": 0.35},
    {"label": "mid", "min_price": 0.35, "max_price": 0.65},
    {"label": "expensive", "min_price": 0.65, "max_price": None},
)
time_to_expiry_buckets = (
    {"label": "final_3m", "min_seconds": None, "max_seconds": 180},
    {"label": "middle_9m", "min_seconds": 180, "max_seconds": 720},
    {"label": "early", "min_seconds": 720, "max_seconds": None},
)
top_policy_price_sub_buckets = (
    {"label": "65_75c", "min_price": 0.65, "max_price": 0.75, "include_max": False},
    {"label": "75_90c", "min_price": 0.75, "max_price": 0.90, "include_max": False},
    {"label": "90_100c", "min_price": 0.90, "max_price": 1.00, "include_max": True},
)
fee_assumption = 0.01
slippage_assumption = 0.00
stress_slippage_assumption = 0.02
cheap_price_max = 0.35
expensive_price_min = 0.65
final_window_seconds = 180
minimum_candidate_trades = 100
minimum_candidate_markets = 10
candlestick_batch_size = 100
rate_limit_sleep_seconds = 0.02

# Compatibility aliases for older notebook checks and quick interactive use.
MARKET_SERIES = market_series
START_DATE = date_start
END_DATE = date_end
SOURCE_MODE = source_mode
DATA_PATH = data_path
OUTPUT_DIR = output_dir
PRICE_BUCKETS = price_buckets
TIME_TO_EXPIRY_BUCKETS = time_to_expiry_buckets
FEE_DOLLARS_PER_CONTRACT = fee_assumption
SLIPPAGE_DOLLARS_PER_CONTRACT = slippage_assumption
STRESS_SLIPPAGE_DOLLARS_PER_CONTRACT = stress_slippage_assumption
REFRESH_SOURCE_DATA_FLAG = REFRESH_SOURCE_DATA
TOP_POLICY_PRICE_SUB_BUCKETS = top_policy_price_sub_buckets
MINIMUM_CANDIDATE_TRADES = minimum_candidate_trades
MINIMUM_CANDIDATE_MARKETS = minimum_candidate_markets

assumption_rows = [
    {"assumption": "market_series", "value": market_series},
    {"assumption": "date_start", "value": date_start.isoformat()},
    {"assumption": "date_end", "value": date_end.isoformat()},
    {"assumption": "source_mode", "value": source_mode},
    {"assumption": "data_path", "value": str(data_path) if data_path else "None"},
    {"assumption": "output_dir", "value": str(output_dir)},
    {"assumption": "source_cache_root", "value": str(source_cache_root)},
    {"assumption": "REFRESH_SOURCE_DATA", "value": REFRESH_SOURCE_DATA},
    {"assumption": "primary_policy", "value": primary_policy},
    {"assumption": "fee_per_contract", "value": fee_assumption},
    {"assumption": "slippage_per_contract", "value": slippage_assumption},
    {"assumption": "stress_slippage_per_contract", "value": stress_slippage_assumption},
    {"assumption": "cheap_price_max", "value": cheap_price_max},
    {"assumption": "expensive_price_min", "value": expensive_price_min},
    {"assumption": "final_window_seconds", "value": final_window_seconds},
    {"assumption": "minimum_candidate_trades", "value": minimum_candidate_trades},
    {"assumption": "minimum_candidate_markets", "value": minimum_candidate_markets},
]

bucket_rows = (
    [{"bucket_type": "price", **bucket} for bucket in price_buckets]
    + [{"bucket_type": "time_to_expiry", **bucket} for bucket in time_to_expiry_buckets]
    + [{"bucket_type": "top_policy_price_sub_bucket", **bucket} for bucket in top_policy_price_sub_buckets]
)

display(pd.DataFrame(assumption_rows))
display(pd.DataFrame(bucket_rows))


,assumption,value
0,market_series,KXBTC15M
1,date_start,2026-06-05
2,date_end,2026-06-05
3,source_mode,kalshi_public
4,data_path,None
5,output_dir,/Users/sid/projects/alphadb/artifacts/market-d...
6,source_cache_root,/Users/sid/projects/alphadb/artifacts/market-d...
7,REFRESH_SOURCE_DATA,False
8,primary_policy,expensive_yes
9,fee_per_contract,0.01


,bucket_type,label,min_price,max_price,min_seconds,max_seconds,include_max
0,price,cheap,NaN,0.35,NaN,NaN,NaN
1,price,mid,0.35,0.65,NaN,NaN,NaN
2,price,expensive,0.65,NaN,NaN,NaN,NaN
3,time_to_expiry,final_3m,NaN,NaN,NaN,180.0,NaN
4,time_to_expiry,middle_9m,NaN,NaN,180.0,720.0,NaN
5,time_to_expiry,early,NaN,NaN,720.0,NaN,NaN
6,top_policy_price_sub_bucket,65_75c,0.65,0.75,NaN,NaN,False
7,top_policy_price_sub_bucket,75_90c,0.75,0.90,NaN,NaN,False
8,top_policy_price_sub_bucket,90_100c,0.90,1.00,NaN,NaN,True


## 3. Data Load + Contract Mechanics

The public-data path uses the same Kalshi market and candlestick contracts as the prior notebook workflow: `GET /markets` for the `KXBTC15M` series over each requested close-date range, then `GET /markets/candlesticks` using one-minute public candlestick payloads. The candlestick call is batched by ticker for speed, but it remains the same public API contract and raw payload shape. Raw source artifacts are cached under ignored `artifacts/` paths so rerunning analysis cells does not repeatedly hit Kalshi.

`KXBTC15M` is a binary event contract. YES pays one dollar when the listed Bitcoin 15-minute up/down condition resolves true, and NO pays when it resolves false. The threshold is read from listed Kalshi market metadata such as `floor_strike`; this notebook does not infer a new threshold from external BTC features. Settlement assumptions follow the repo settlement note: CF Benchmarks BRTI is averaged over the final 60 seconds before the listed expiration window, with official Kalshi settlement fields used as labels in this public-data workbench.


In [2]:
KALSHI_PUBLIC_BASE_URL = "https://external-api.kalshi.com/trade-api/v2"


def _date_start_ts(value: date) -> int:
    return int(datetime.combine(value, datetime_time.min, tzinfo=UTC).timestamp())


def _date_end_ts(value: date) -> int:
    return int(datetime.combine(value, datetime_time.max, tzinfo=UTC).timestamp())


def _parse_dt(value: Any) -> datetime | None:
    if value in (None, ""):
        return None
    if isinstance(value, (int, float)):
        return datetime.fromtimestamp(float(value), tz=UTC)
    text = str(value)
    try:
        return datetime.fromisoformat(text.replace("Z", "+00:00")).astimezone(UTC)
    except ValueError:
        return None


def _iso(value: datetime | None) -> str | None:
    if value is None:
        return None
    return value.astimezone(UTC).isoformat().replace("+00:00", "Z")


def _float(value: Any) -> float | None:
    if value in (None, ""):
        return None
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _prob_from_nested(payload: dict[str, Any], key: str) -> float | None:
    value = payload.get(key)
    if isinstance(value, dict):
        for nested_key in ("close_dollars", "close", "value"):
            numeric = _float(value.get(nested_key))
            if numeric is not None:
                return numeric
    return _float(value)


def _market_ticker(market: dict[str, Any]) -> str:
    return str(market.get("ticker") or market.get("market_ticker") or "")


def _kalshi_get(path: str, params: dict[str, str]) -> dict[str, Any]:
    url = f"{KALSHI_PUBLIC_BASE_URL}{path}?{parse.urlencode(params)}"
    req = request.Request(url, headers={"Accept": "application/json", "User-Agent": "alphadb-notebook/0.1"})
    with request.urlopen(req, timeout=45) as response:
        return json.loads(response.read().decode("utf-8"))


def _window_start(end: date, days: int) -> date:
    return end - timedelta(days=days - 1)


def _expected_day_strings(start: date, end: date) -> list[str]:
    return [(start + timedelta(days=offset)).isoformat() for offset in range((end - start).days + 1)]


def _window_cache_path(window_label: str, start: date, end: date) -> Path:
    safe = f"{market_series.lower()}-{window_label}-{start.isoformat()}-{end.isoformat()}"
    return source_cache_root / safe / "source.json"


def _list_public_markets(start: date, end: date) -> tuple[list[dict[str, Any]], list[str]]:
    markets: list[dict[str, Any]] = []
    request_paths: list[str] = []
    cursor = ""
    while True:
        params = {
            "series_ticker": market_series,
            "min_close_ts": str(_date_start_ts(start)),
            "max_close_ts": str(_date_end_ts(end)),
            "mve_filter": "exclude",
            "limit": "1000",
        }
        if cursor:
            params["cursor"] = cursor
        request_paths.append(f"/markets?{parse.urlencode(params)}")
        payload = _kalshi_get("/markets", params)
        markets.extend(payload.get("markets") or [])
        cursor = str(payload.get("cursor") or "")
        if not cursor:
            return markets, request_paths


def _market_start_end_ts(market: dict[str, Any], default_start: date, default_end: date) -> tuple[int, int]:
    start_dt = _parse_dt(market.get("open_time") or market.get("open_ts"))
    end_dt = _parse_dt(market.get("close_time") or market.get("expected_expiration_time") or market.get("occurrence_datetime") or market.get("expiration_time"))
    start_ts = int(start_dt.timestamp()) if start_dt else _date_start_ts(default_start)
    end_ts = int(end_dt.timestamp()) if end_dt else _date_end_ts(default_end)
    return start_ts, end_ts


def _chunks(values: list[str], size: int) -> list[list[str]]:
    return [values[index : index + size] for index in range(0, len(values), size)]


def _estimated_candlestick_count(ranges: list[tuple[int, int]], period_seconds: int = 60) -> int:
    if not ranges:
        return 0
    start_ts = min(start for start, _ in ranges)
    end_ts = max(end for _, end in ranges)
    return len(ranges) * max(1, ((end_ts - start_ts) // period_seconds) + 1)


def _candlestick_batches(tickers: list[str], markets_by_ticker: dict[str, dict[str, Any]], start: date, end: date) -> list[list[str]]:
    batches: list[list[str]] = []
    batch: list[str] = []
    batch_ranges: list[tuple[int, int]] = []
    max_candles_per_request = 9000
    for ticker in tickers:
        market_range = _market_start_end_ts(markets_by_ticker[ticker], start, end)
        candidate_ranges = [*batch_ranges, market_range]
        oversized = _estimated_candlestick_count(candidate_ranges) > max_candles_per_request
        too_many_tickers = len(candidate_ranges) > candlestick_batch_size
        if batch and (oversized or too_many_tickers):
            batches.append(batch)
            batch = [ticker]
            batch_ranges = [market_range]
        else:
            batch.append(ticker)
            batch_ranges = candidate_ranges
    if batch:
        batches.append(batch)
    return batches


def _fetch_public_window(window_label: str, start: date, end: date) -> tuple[list[dict[str, Any]], dict[str, list[dict[str, Any]]], dict[str, Any]]:
    markets, request_paths = _list_public_markets(start, end)
    tickers = [_market_ticker(market) for market in markets if _market_ticker(market)]
    candles_by_market: dict[str, list[dict[str, Any]]] = {ticker: [] for ticker in tickers}
    if tickers:
        markets_by_ticker = {_market_ticker(market): market for market in markets if _market_ticker(market)}
        for ticker_chunk in _candlestick_batches(tickers, markets_by_ticker, start, end):
            chunk_ranges = [_market_start_end_ts(markets_by_ticker[ticker], start, end) for ticker in ticker_chunk]
            # The public API caps each request by total candles across all markets, so the batch
            # window must be local to this ticker chunk and below the request cap.
            batch_start_ts = min(start_ts for start_ts, _ in chunk_ranges)
            batch_end_ts = max(end_ts for _, end_ts in chunk_ranges)
            params = {
                "market_tickers": ",".join(ticker_chunk),
                "start_ts": str(batch_start_ts),
                "end_ts": str(batch_end_ts),
                "period_interval": "1",
            }
            request_paths.append(f"/markets/candlesticks?{parse.urlencode(params)}")
            payload = _kalshi_get("/markets/candlesticks", params)
            for item in payload.get("markets") or []:
                ticker = item.get("market_ticker") or item.get("ticker")
                if ticker:
                    candles_by_market[str(ticker)] = list(item.get("candlesticks") or [])
            if rate_limit_sleep_seconds:
                time.sleep(rate_limit_sleep_seconds)
    provenance = {
        "source_mode": "kalshi_public",
        "source_status": "fresh_public_pull",
        "retrieved_at": _iso(datetime.now(UTC)),
        "window": window_label,
        "market_series": market_series,
        "start_date": start.isoformat(),
        "end_date": end.isoformat(),
        "expected_days": _expected_day_strings(start, end),
        "market_identifiers": tickers,
        "source_files": [],
        "request_paths": request_paths,
    }
    return markets, candles_by_market, provenance


def _read_source_cache(path: Path) -> tuple[list[dict[str, Any]], dict[str, list[dict[str, Any]]], dict[str, Any]]:
    payload = json.loads(path.read_text(encoding="utf-8"))
    markets = list(payload.get("markets") or [])
    candles_by_market = {
        item.get("market_ticker"): list(item.get("candlesticks") or [])
        for item in payload.get("candlesticks_by_market", [])
        if item.get("market_ticker")
    }
    provenance = dict(payload.get("provenance") or {})
    provenance["source_status"] = "cached_source"
    provenance["source_files"] = sorted(set([*provenance.get("source_files", []), str(path)]))
    return markets, candles_by_market, provenance


def _write_source_cache(path: Path, markets: list[dict[str, Any]], candles_by_market: dict[str, list[dict[str, Any]]], provenance: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "schema_version": "kxbtc15m_public_source_cache.v1",
        "created_at": _iso(datetime.now(UTC)),
        "provenance": {**provenance, "source_files": [str(path)]},
        "markets": markets,
        "candlesticks_by_market": [
            {"market_ticker": ticker, "candlesticks": rows}
            for ticker, rows in sorted(candles_by_market.items())
        ],
    }
    path.write_text(json.dumps(payload, indent=2) + "\n", encoding="utf-8")


def _load_window_source(window_label: str, start: date, end: date) -> tuple[list[dict[str, Any]], dict[str, list[dict[str, Any]]], dict[str, Any]]:
    if source_mode != "kalshi_public":
        raise ValueError("Exact current/7d/30d comparison requires source_mode='kalshi_public'.")
    cache_path = _window_cache_path(window_label, start, end)
    if cache_path.exists() and not REFRESH_SOURCE_DATA:
        return _read_source_cache(cache_path)
    try:
        markets, candles_by_market, provenance = _fetch_public_window(window_label, start, end)
    except Exception as exc:
        if cache_path.exists() and not REFRESH_SOURCE_DATA:
            return _read_source_cache(cache_path)
        raise RuntimeError(f"Failed to load required {window_label} source window {start}:{end}: {exc}") from exc
    _write_source_cache(cache_path, markets, candles_by_market, provenance)
    provenance = {**provenance, "source_files": [str(cache_path)]}
    return markets, candles_by_market, provenance


def _outcome_yes(market: dict[str, Any]) -> tuple[int | None, str]:
    for key in ("result", "outcome", "settlement_value"):
        value = market.get(key)
        if isinstance(value, str):
            lower = value.lower()
            if lower in {"yes", "true", "1"}:
                return 1, "settled"
            if lower in {"no", "false", "0"}:
                return 0, "settled"
    settlement_value = _float(market.get("yes_settlement_value_dollars") or market.get("yes_settlement_value") or market.get("settlement_value_dollars"))
    if settlement_value is not None:
        if settlement_value >= 0.99:
            return 1, "settled"
        if settlement_value <= 0.01:
            return 0, "settled"
        return None, "missing"
    return (None, "missing") if str(market.get("status") or "").lower() == "settled" else (None, "unresolved")


def _price_bucket(value: Any) -> str:
    numeric = _float(value)
    for bucket in price_buckets:
        low = bucket["min_price"]
        high = bucket["max_price"]
        if numeric is not None and (low is None or numeric >= low) and (high is None or numeric < high):
            return bucket["label"]
    return "missing"


def _top_policy_price_sub_bucket(value: Any) -> str:
    numeric = _float(value)
    for bucket in top_policy_price_sub_buckets:
        low = bucket["min_price"]
        high = bucket["max_price"]
        include_max = bool(bucket.get("include_max"))
        if numeric is None or numeric < low:
            continue
        if (include_max and numeric <= high) or (not include_max and numeric < high):
            return bucket["label"]
    return "outside_top_policy_range"


def _time_to_expiry_bucket(value: Any) -> str:
    numeric = None if value is None else int(value)
    for bucket in time_to_expiry_buckets:
        low = bucket["min_seconds"]
        high = bucket["max_seconds"]
        if numeric is not None and (low is None or numeric >= low) and (high is None or numeric < high):
            return bucket["label"]
    return "missing"


def _normalize_rows(markets: list[dict[str, Any]], candles_by_market: dict[str, list[dict[str, Any]]], provenance: dict[str, Any]) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    for market in markets:
        ticker = _market_ticker(market)
        open_dt = _parse_dt(market.get("open_time") or market.get("open_ts"))
        close_dt = _parse_dt(market.get("close_time") or market.get("expected_expiration_time") or market.get("expiration_time"))
        outcome_yes, outcome_state = _outcome_yes(market)
        payout_threshold = _float(market.get("floor_strike") or market.get("cap_strike") or market.get("strike") or market.get("payout_threshold"))
        for candle in candles_by_market.get(ticker, []):
            decision_dt = _parse_dt(candle.get("end_period_ts") or candle.get("end_time")) or open_dt
            time_to_expiry_seconds = int((close_dt - decision_dt).total_seconds()) if close_dt and decision_dt else None
            yes_bid = _prob_from_nested(candle, "yes_bid")
            yes_ask = _prob_from_nested(candle, "yes_ask")
            no_bid = _prob_from_nested(candle, "no_bid")
            no_ask = _prob_from_nested(candle, "no_ask")
            no_price_source = "explicit"
            if no_bid is None and yes_ask is not None:
                no_bid = round(1.0 - yes_ask, 10)
                no_price_source = "derived_binary_complement"
            if no_ask is None and yes_bid is not None:
                no_ask = round(1.0 - yes_bid, 10)
                no_price_source = "derived_binary_complement"
            base = {
                "market_series": market_series,
                "market_ticker": ticker,
                "event_ticker": market.get("event_ticker"),
                "market_status": market.get("status"),
                "market_open_time": _iso(open_dt),
                "market_close_time": _iso(close_dt),
                "market_close_date": close_dt.date().isoformat() if close_dt else None,
                "decision_timestamp": _iso(decision_dt),
                "decision_date": decision_dt.date().isoformat() if decision_dt else None,
                "time_to_expiry_seconds": time_to_expiry_seconds,
                "yes_bid": yes_bid,
                "yes_ask": yes_ask,
                "no_bid": no_bid,
                "no_ask": no_ask,
                "no_price_source": no_price_source,
                "last_price": _prob_from_nested(candle, "price") or _prob_from_nested(candle, "last_price"),
                "volume": _float(candle.get("volume_fp") or candle.get("volume")),
                "open_interest": _float(candle.get("open_interest_fp") or candle.get("open_interest")),
                "payout_threshold": payout_threshold,
                "outcome_yes": outcome_yes,
                "outcome_state": outcome_state,
                "settlement_label_source": "kalshi_settlement_fields",
                "provenance_source_mode": provenance["source_mode"],
                "source_status": provenance.get("source_status"),
            }
            for side in ("YES", "NO"):
                side_bid = yes_bid if side == "YES" else no_bid
                side_ask = yes_ask if side == "YES" else no_ask
                side_outcome = outcome_yes if side == "YES" else (None if outcome_yes is None else int(not bool(outcome_yes)))
                reasons = []
                if side_ask is None:
                    reasons.append("missing_executable_ask")
                if outcome_state != "settled":
                    reasons.append("missing_outcome")
                if side_ask is not None and not (0.0 <= float(side_ask) <= 1.0):
                    reasons.append("invalid_probability")
                row = {
                    **base,
                    "side": side,
                    "side_bid": side_bid,
                    "side_ask": side_ask,
                    "side_spread": None if side_bid is None or side_ask is None else side_ask - side_bid,
                    "side_outcome": side_outcome,
                    "exclusion_reasons": sorted(set(reasons)),
                }
                row["price_bucket"] = _price_bucket(row["side_ask"])
                row["top_policy_price_sub_bucket"] = _top_policy_price_sub_bucket(row["side_ask"])
                row["time_to_expiry_bucket"] = _time_to_expiry_bucket(row["time_to_expiry_seconds"])
                rows.append(row)
    key_counts = Counter((row["market_ticker"], row["decision_timestamp"], row["side"]) for row in rows)
    for row in rows:
        key = (row["market_ticker"], row["decision_timestamp"], row["side"])
        if key_counts[key] > 1:
            row["exclusion_reasons"] = sorted(set(row["exclusion_reasons"] + ["duplicate_candidate"]))
    return pd.DataFrame(rows)


def _validate_complete_window(window_label: str, frame: pd.DataFrame, start: date, end: date) -> None:
    expected = set(_expected_day_strings(start, end))
    if frame.empty:
        raise RuntimeError(f"Required {window_label} window has no normalized rows.")
    settled_days = set(frame.loc[frame["outcome_state"].eq("settled"), "market_close_date"].dropna().unique())
    missing = sorted(expected - settled_days)
    if missing:
        raise RuntimeError(f"Required {window_label} window is incomplete; missing settled market close days: {missing}")


def contract_mechanics_rows() -> list[dict[str, Any]]:
    try:
        from alphadb.markets.registry import default_market_registry

        spec = default_market_registry().get(market_series)
        final_window = spec.settlement_spec.final_settlement_window if spec.settlement_spec else None
        comparator_rules = spec.settlement_spec.payout_comparator_rules if spec.settlement_spec else []
        comparator_text = "; ".join(rule.description for rule in comparator_rules)
        return [
            {"field": "contract_type", "value": "Binary event contract"},
            {"field": "yes_meaning", "value": "YES pays when the listed KXBTC15M condition resolves true."},
            {"field": "no_meaning", "value": "NO pays when the listed condition resolves false."},
            {"field": "underlying", "value": spec.underlying},
            {"field": "btc_threshold", "value": "Listed market metadata, usually floor_strike."},
            {"field": "settlement_source", "value": spec.settlement_source},
            {"field": "settlement_window", "value": f"{final_window.duration_seconds}s ending at market close" if final_window else "not available"},
            {"field": "comparator_semantics", "value": comparator_text},
            {"field": "label_source", "value": spec.label_function.outcome_source},
        ]
    except Exception as exc:
        return [
            {"field": "contract_type", "value": "Binary event contract"},
            {"field": "yes_meaning", "value": "YES pays when the listed KXBTC15M condition resolves true."},
            {"field": "no_meaning", "value": "NO pays when the listed condition resolves false."},
            {"field": "btc_threshold", "value": "Listed market metadata, usually floor_strike."},
            {"field": "settlement_source", "value": "CF Benchmarks RTI"},
            {"field": "settlement_window", "value": "Final 60 seconds before listed expiration"},
            {"field": "label_source", "value": f"Kalshi settlement fields; registry import failed: {exc}"},
        ]

window_runs: dict[str, dict[str, Any]] = {}
for spec in window_specs:
    window_label = spec["window"]
    window_start = _window_start(date_end, spec["days"])
    markets, candles_by_market, source_provenance = _load_window_source(window_label, window_start, date_end)
    window_rows_df = _normalize_rows(markets, candles_by_market, source_provenance)
    _validate_complete_window(window_label, window_rows_df, window_start, date_end)
    window_runs[window_label] = {
        "window": window_label,
        "start": window_start,
        "end": date_end,
        "expected_days": _expected_day_strings(window_start, date_end),
        "markets": markets,
        "candles_by_market": candles_by_market,
        "source_provenance": source_provenance,
        "rows": window_rows_df,
    }

current_run = window_runs["current"]
markets = current_run["markets"]
candles_by_market = current_run["candles_by_market"]
source_provenance = current_run["source_provenance"]
rows_df = current_run["rows"]
contract_mechanics_df = pd.DataFrame(contract_mechanics_rows())

window_source_summary_df = pd.DataFrame([
    {
        "window": run["window"],
        "start": run["start"].isoformat(),
        "end": run["end"].isoformat(),
        "expected_days": len(run["expected_days"]),
        "market_close_days": run["rows"]["market_close_date"].nunique(),
        "markets_loaded": len(run["markets"]),
        "candlestick_rows_loaded": sum(len(rows) for rows in run["candles_by_market"].values()),
        "normalized_side_rows": len(run["rows"]),
        "source_status": run["source_provenance"].get("source_status"),
        "source_file": ", ".join(run["source_provenance"].get("source_files", [])),
    }
    for run in window_runs.values()
])

data_load_summary_df = pd.DataFrame([
    {"metric": "current_markets_loaded", "value": len(markets)},
    {"metric": "current_candlestick_markets_loaded", "value": len(candles_by_market)},
    {"metric": "current_candlestick_rows_loaded", "value": sum(len(rows) for rows in candles_by_market.values())},
    {"metric": "current_normalized_side_rows", "value": len(rows_df)},
    {"metric": "source_mode", "value": source_provenance["source_mode"]},
    {"metric": "source_status", "value": source_provenance.get("source_status")},
    {"metric": "retrieved_at", "value": source_provenance["retrieved_at"]},
])

display(window_source_summary_df)
display(data_load_summary_df)
display(contract_mechanics_df)


,window,start,end,expected_days,market_close_days,markets_loaded,candlestick_rows_loaded,normalized_side_rows,source_status,source_file
0,current,2026-06-05,2026-06-05,1,1,96,1532,3064,cached_source,/Users/sid/projects/alphadb/artifacts/market-d...
1,7d,2026-05-30,2026-06-05,7,7,660,10532,21064,cached_source,/Users/sid/projects/alphadb/artifacts/market-d...
2,30d,2026-05-07,2026-06-05,30,30,2820,44961,89922,cached_source,/Users/sid/projects/alphadb/artifacts/market-d...


,metric,value
0,current_markets_loaded,96
1,current_candlestick_markets_loaded,96
2,current_candlestick_rows_loaded,1532
3,current_normalized_side_rows,3064
4,source_mode,kalshi_public
5,source_status,cached_source
6,retrieved_at,2026-06-07T16:17:49.882058Z


,field,value
0,contract_type,Binary event contract
1,yes_meaning,YES pays when the listed KXBTC15M condition re...
2,no_meaning,NO pays when the listed condition resolves false.
3,underlying,BTC
4,btc_threshold,"Listed market metadata, usually floor_strike."
5,settlement_source,CF Benchmarks RTI
6,settlement_window,60s ending at market close
7,comparator_semantics,Expiration value must be strictly greater than...
8,label_source,kalshi_settlement


## 4. Data Quality Audit

A compact audit answers whether the current slice is broad enough to interpret at all: row counts, market counts, coverage, missing executable prices, missing outcomes, and excluded rows. Exact-window completeness for current / 7d / 30d is validated before this table is displayed.


In [3]:
def build_data_quality_audit(frame: pd.DataFrame) -> pd.DataFrame:
    exclusions = Counter(reason for reasons in frame["exclusion_reasons"] for reason in reasons)
    close_dates = sorted(x for x in frame["market_close_date"].dropna().unique())
    decision_dates = sorted(x for x in frame["decision_date"].dropna().unique())
    return pd.DataFrame([
        {"metric": "row_count", "value": len(frame), "detail": "normalized side rows"},
        {"metric": "market_count", "value": frame["market_ticker"].nunique(), "detail": "unique markets"},
        {"metric": "market_close_date_range", "value": f"{close_dates[0]}:{close_dates[-1]}" if close_dates else "", "detail": "market close date coverage"},
        {"metric": "decision_date_range", "value": f"{decision_dates[0]}:{decision_dates[-1]}" if decision_dates else "", "detail": "decision timestamp coverage"},
        {"metric": "unique_market_close_days", "value": len(close_dates), "detail": ", ".join(close_dates)},
        {"metric": "missing_prices", "value": int(frame["last_price"].isna().sum()), "detail": "missing last/price close"},
        {"metric": "missing_asks", "value": int(frame["side_ask"].isna().sum()), "detail": "missing executable side ask"},
        {"metric": "missing_outcomes", "value": int(frame["side_outcome"].isna().sum()), "detail": "unsettled or missing labels"},
        {"metric": "excluded_rows", "value": int(frame["exclusion_reasons"].map(bool).sum()), "detail": json.dumps(dict(sorted(exclusions.items())))},
    ])

data_quality_audit_df = build_data_quality_audit(rows_df)
data_quality_audit_df


,metric,value,detail
0,row_count,3064,normalized side rows
1,market_count,96,unique markets
2,market_close_date_range,2026-06-05:2026-06-05,market close date coverage
3,decision_date_range,2026-06-04:2026-06-05,decision timestamp coverage
4,unique_market_close_days,1,2026-06-05
5,missing_prices,186,missing last/price close
6,missing_asks,0,missing executable side ask
7,missing_outcomes,0,unsettled or missing labels
8,excluded_rows,0,{}


## 5. Market Structure Segmentation

These tables describe the current market slice before treating any pattern as edge. They segment by side, price bucket, time-to-expiry bucket, side plus price, and side plus price plus time-to-expiry.


In [4]:
def segment_summary(frame: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    grouped = frame.groupby(keys, dropna=False)
    out = grouped.agg(
        rows=("market_ticker", "size"),
        markets=("market_ticker", "nunique"),
        market_close_days=("market_close_date", "nunique"),
        settled_rows=("side_outcome", lambda s: int(s.notna().sum())),
        missing_asks=("side_ask", lambda s: int(s.isna().sum())),
        avg_entry_ask=("side_ask", "mean"),
        realized_frequency=("side_outcome", "mean"),
    ).reset_index()
    return out.sort_values(keys).reset_index(drop=True)

segmentation_tables = {
    "side": segment_summary(rows_df, ["side"]),
    "price_bucket": segment_summary(rows_df, ["price_bucket"]),
    "time_to_expiry_bucket": segment_summary(rows_df, ["time_to_expiry_bucket"]),
    "side_price_bucket": segment_summary(rows_df, ["side", "price_bucket"]),
    "side_price_time_bucket": segment_summary(rows_df, ["side", "price_bucket", "time_to_expiry_bucket"]),
}

for name, table in segmentation_tables.items():
    display(Markdown(f"**{name}**"))
    display(table)


**side**

,side,rows,markets,market_close_days,settled_rows,missing_asks,avg_entry_ask,realized_frequency
0,NO,1532,96,1,1532,0,0.574804,0.530679
1,YES,1532,96,1,1532,0,0.492409,0.469321


**price_bucket**

,price_bucket,rows,markets,market_close_days,settled_rows,missing_asks,avg_entry_ask,realized_frequency
0,cheap,1109,96,1,1109,0,0.123994,0.093778
1,expensive,1330,96,1,1330,0,0.892437,0.845113
2,mid,625,86,1,625,0,0.496832,0.486400


**time_to_expiry_bucket**

,time_to_expiry_bucket,rows,markets,market_close_days,settled_rows,missing_asks,avg_entry_ask,realized_frequency
0,early,576,96,1,576,0,0.505517,0.5
1,final_3m,760,96,1,760,0,0.622067,0.5
2,middle_9m,1728,96,1,1728,0,0.504063,0.5


**side_price_bucket**

,side,price_bucket,rows,markets,market_close_days,settled_rows,missing_asks,avg_entry_ask,realized_frequency
0,NO,cheap,478,58,1,478,0,0.126933,0.058577
1,NO,expensive,746,95,1,746,0,0.891776,0.829759
2,NO,mid,308,83,1,308,0,0.502143,0.538961
3,YES,cheap,631,70,1,631,0,0.121767,0.120444
4,YES,expensive,584,93,1,584,0,0.893281,0.864726
5,YES,mid,317,86,1,317,0,0.491672,0.435331


**side_price_time_bucket**

,side,price_bucket,time_to_expiry_bucket,rows,markets,market_close_days,settled_rows,missing_asks,avg_entry_ask,realized_frequency
0,NO,cheap,early,49,29,1,49,0,0.263510,0.142857
1,NO,cheap,final_3m,125,45,1,125,0,0.019424,0.000000
2,NO,cheap,middle_9m,304,51,1,304,0,0.149125,0.069079
3,NO,expensive,early,93,46,1,93,0,0.744516,0.709677
4,NO,expensive,final_3m,249,95,1,249,0,0.984843,0.803213
5,NO,expensive,middle_9m,404,64,1,404,0,0.868314,0.873762
6,NO,mid,early,146,75,1,146,0,0.503288,0.547945
7,NO,mid,final_3m,6,5,1,6,0,0.505000,0.166667
8,NO,mid,middle_9m,156,56,1,156,0,0.500962,0.544872
9,YES,cheap,early,83,45,1,83,0,0.255422,0.277108


## 6. Dumb Baselines

These are deliberately unsophisticated one-contract baselines. They are not Strategy Specs or live strategy recommendations. They ask whether the market structure itself shows obvious pockets before any model, agent, or live-trading machinery enters the picture.


In [5]:
def _valid_trade_rows(frame: pd.DataFrame) -> pd.DataFrame:
    valid = frame.copy()
    valid = valid[valid["side_ask"].notna() & valid["side_outcome"].notna()]
    valid = valid[~valid["exclusion_reasons"].map(lambda reasons: "invalid_probability" in reasons or "duplicate_candidate" in reasons)]
    return valid


def _simulate_policy(frame: pd.DataFrame, policy: str, selector) -> tuple[dict[str, Any], pd.DataFrame]:
    selected = frame[selector(frame)].copy()
    valid = _valid_trade_rows(selected).copy()
    if valid.empty:
        summary = {
            "policy": policy,
            "selected_rows": len(selected),
            "trades": 0,
            "markets": 0,
            "unique_days": 0,
            "net_pnl": 0.0,
            "net_roi": None,
            "stressed_roi": None,
            "excluded_rows": len(selected),
        }
        return summary, pd.DataFrame()
    valid["policy"] = policy
    valid["contracts"] = 1
    valid["payout"] = valid["side_outcome"].astype(float)
    valid["gross_pnl"] = valid["payout"] - valid["side_ask"].astype(float)
    valid["fee"] = fee_assumption
    valid["slippage"] = slippage_assumption
    valid["stress_slippage"] = stress_slippage_assumption
    valid["net_pnl"] = valid["gross_pnl"] - valid["fee"] - valid["slippage"]
    valid["stress_pnl"] = valid["gross_pnl"] - valid["fee"] - valid["stress_slippage"]
    valid["cost_basis"] = valid["side_ask"].astype(float) + valid["fee"] + valid["slippage"]
    valid["stress_cost_basis"] = valid["side_ask"].astype(float) + valid["fee"] + valid["stress_slippage"]
    net_pnl = float(valid["net_pnl"].sum())
    stress_pnl = float(valid["stress_pnl"].sum())
    cost_basis = float(valid["cost_basis"].sum())
    stress_cost_basis = float(valid["stress_cost_basis"].sum())
    summary = {
        "policy": policy,
        "selected_rows": len(selected),
        "trades": len(valid),
        "markets": valid["market_ticker"].nunique(),
        "unique_days": valid["market_close_date"].nunique(),
        "net_pnl": net_pnl,
        "net_roi": net_pnl / cost_basis if cost_basis else None,
        "stressed_roi": stress_pnl / stress_cost_basis if stress_cost_basis else None,
        "excluded_rows": len(selected) - len(valid),
    }
    return summary, valid

policy_specs = [
    ("buy_every_yes", lambda f: f["side"].eq("YES")),
    ("buy_every_no", lambda f: f["side"].eq("NO")),
    ("cheap_yes", lambda f: f["side"].eq("YES") & f["side_ask"].le(cheap_price_max)),
    ("cheap_no", lambda f: f["side"].eq("NO") & f["side_ask"].le(cheap_price_max)),
    ("expensive_yes", lambda f: f["side"].eq("YES") & f["side_ask"].ge(expensive_price_min)),
    ("expensive_no", lambda f: f["side"].eq("NO") & f["side_ask"].ge(expensive_price_min)),
    ("final_window_yes", lambda f: f["side"].eq("YES") & f["time_to_expiry_seconds"].le(final_window_seconds)),
    ("final_window_no", lambda f: f["side"].eq("NO") & f["time_to_expiry_seconds"].le(final_window_seconds)),
]


def run_baseline_set(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    baseline_rows = []
    trade_frames = []
    for policy, selector in policy_specs:
        summary, trades = _simulate_policy(frame, policy, selector)
        baseline_rows.append(summary)
        if not trades.empty:
            trade_frames.append(trades)
    baseline = pd.DataFrame(baseline_rows).sort_values("net_pnl", ascending=False).reset_index(drop=True)
    trades = pd.concat(trade_frames, ignore_index=True) if trade_frames else pd.DataFrame()
    return baseline, trades

baseline_results_df, all_trades_df = run_baseline_set(rows_df)
window_baselines: dict[str, pd.DataFrame] = {}
window_trades: dict[str, pd.DataFrame] = {}
for label, run in window_runs.items():
    window_baselines[label], window_trades[label] = run_baseline_set(run["rows"])

baseline_results_df


,policy,selected_rows,trades,markets,unique_days,net_pnl,net_roi,stressed_roi,excluded_rows
0,cheap_yes,640,640,71,1,-7.385,-0.085489,-0.203509,0
1,expensive_yes,584,584,93,1,-22.516,-0.042683,-0.063420,0
2,cheap_no,482,482,58,1,-37.894,-0.566478,-0.621083,0
3,final_window_yes,476,476,96,1,-45.155,-0.167766,-0.196196,0
4,buy_every_yes,1532,1532,96,1,-50.691,-0.065859,-0.101622,0
5,expensive_no,746,746,95,1,-53.725,-0.079862,-0.099826,0
6,final_window_no,476,476,96,1,-57.619,-0.186096,-0.210375,0
7,buy_every_no,1532,1532,96,1,-82.919,-0.092552,-0.122560,0


### Exact-Window Comparison

The primary comparison tracks `expensive_yes` across exact current / 7d / 30d windows ending at `date_end`. Each window must include every expected market-close calendar day. The best policy per window is shown only as a secondary sanity check so the notebook does not quietly turn into a winner-picking exercise.


In [6]:
def _policy_row_or_empty(baseline: pd.DataFrame, policy: str) -> pd.Series:
    match = baseline[baseline["policy"].eq(policy)]
    if match.empty:
        raise RuntimeError(f"Required policy {policy} is missing from baseline results.")
    return match.iloc[0]

window_comparison_rows = []
for label, run in window_runs.items():
    baseline = window_baselines[label]
    primary_row = _policy_row_or_empty(baseline, primary_policy)
    if int(primary_row["trades"]) == 0:
        raise RuntimeError(f"Required window {label} has zero {primary_policy} trades.")
    least_bad_row = baseline.sort_values(["net_pnl", "net_roi"], ascending=False).iloc[0]
    least_bad_positive = bool(least_bad_row["net_roi"] > 0 and least_bad_row["stressed_roi"] > 0)
    source_status = run["source_provenance"].get("source_status")
    window_comparison_rows.append({
        "window": label,
        "start": run["start"].isoformat(),
        "end": run["end"].isoformat(),
        "expected_days": len(run["expected_days"]),
        "observed_market_close_days": run["rows"]["market_close_date"].nunique(),
        "source_status": source_status,
        "primary_policy": primary_policy,
        "primary_trades": int(primary_row["trades"]),
        "primary_markets": int(primary_row["markets"]),
        "primary_net_pnl": primary_row["net_pnl"],
        "primary_net_roi": primary_row["net_roi"],
        "primary_stressed_roi": primary_row["stressed_roi"],
        "least_bad_policy": least_bad_row["policy"],
        "least_bad_policy_positive": least_bad_positive,
        "least_bad_policy_net_pnl": least_bad_row["net_pnl"],
        "least_bad_policy_net_roi": least_bad_row["net_roi"],
        "least_bad_policy_stressed_roi": least_bad_row["stressed_roi"],
    })

window_comparison_df = pd.DataFrame(window_comparison_rows)
window_comparison_df


,window,start,end,expected_days,observed_market_close_days,source_status,primary_policy,primary_trades,primary_markets,primary_net_pnl,primary_net_roi,primary_stressed_roi,least_bad_policy,least_bad_policy_positive,least_bad_policy_net_pnl,least_bad_policy_net_roi,least_bad_policy_stressed_roi
0,current,2026-06-05,2026-06-05,1,1,cached_source,expensive_yes,584,93,-22.516,-0.042683,-0.063420,cheap_yes,False,-7.385,-0.085489,-0.203509
1,7d,2026-05-30,2026-06-05,7,7,cached_source,expensive_yes,4180,649,-407.772,-0.108140,-0.127484,cheap_no,False,-38.680,-0.080469,-0.196364
2,30d,2026-05-07,2026-06-05,30,30,cached_source,expensive_yes,18220,2749,-1706.896,-0.104398,-0.123924,cheap_no,False,-95.485,-0.044033,-0.161464


## 7. Calibration Analysis

Calibration compares entry ask-implied probability against realized frequency. Positive calibration error means realized frequency exceeded the average entry ask for the group; negative means the group under-realized relative to price. The ROI columns apply the same one-contract friction assumptions to each group.


In [7]:
def _group_roi(frame: pd.DataFrame) -> tuple[float | None, float | None]:
    valid = _valid_trade_rows(frame).copy()
    if valid.empty:
        return None, None
    gross = valid["side_outcome"].astype(float) - valid["side_ask"].astype(float)
    net = gross - fee_assumption - slippage_assumption
    stress = gross - fee_assumption - stress_slippage_assumption
    cost = valid["side_ask"].astype(float) + fee_assumption + slippage_assumption
    stress_cost = valid["side_ask"].astype(float) + fee_assumption + stress_slippage_assumption
    return float(net.sum() / cost.sum()), float(stress.sum() / stress_cost.sum())


def calibration_table(frame: pd.DataFrame, name: str, keys: list[str]) -> pd.DataFrame:
    rows = []
    for key_values, group in frame.groupby(keys, dropna=False):
        if not isinstance(key_values, tuple):
            key_values = (key_values,)
        settled = group[group["side_outcome"].notna() & group["side_ask"].notna()]
        realized = settled["side_outcome"].mean() if not settled.empty else None
        implied = settled["side_ask"].mean() if not settled.empty else None
        net_roi, stressed_roi = _group_roi(group)
        row = {
            "table": name,
            "rows": len(group),
            "settled_rows": len(settled),
            "implied_probability": implied,
            "realized_frequency": realized,
            "calibration_error": None if realized is None or implied is None else realized - implied,
            "net_roi": net_roi,
            "stressed_roi": stressed_roi,
        }
        row.update(dict(zip(keys, key_values, strict=True)))
        rows.append(row)
    return pd.DataFrame(rows)

calibration_results_df = pd.concat(
    [
        calibration_table(rows_df, "side", ["side"]),
        calibration_table(rows_df, "price_bucket", ["price_bucket"]),
        calibration_table(rows_df, "time_to_expiry_bucket", ["time_to_expiry_bucket"]),
        calibration_table(rows_df, "side_price_time_bucket", ["side", "price_bucket", "time_to_expiry_bucket"]),
    ],
    ignore_index=True,
).sort_values(["table", "calibration_error"], ascending=[True, False], na_position="last")

calibration_results_df


,table,rows,settled_rows,implied_probability,realized_frequency,calibration_error,net_roi,stressed_roi,side,price_bucket,time_to_expiry_bucket
4,price_bucket,625,625,0.496832,0.486400,-0.010432,-0.040313,-0.076746,NaN,mid,NaN
2,price_bucket,1109,1109,0.123994,0.093778,-0.030216,-0.300130,-0.391026,NaN,cheap,NaN
3,price_bucket,1330,1330,0.892437,0.845113,-0.047324,-0.063521,-0.083826,NaN,expensive,NaN
1,side,1532,1532,0.492409,0.469321,-0.023088,-0.065859,-0.101622,YES,NaN,NaN
0,side,1532,1532,0.574804,0.530679,-0.044125,-0.092552,-0.122560,NO,NaN,NaN
24,side_price_time_bucket,8,8,0.468750,0.750000,0.281250,0.566580,0.503759,YES,mid,final_3m
20,side_price_time_bucket,56,56,0.735464,0.857143,0.121679,0.149811,0.119769,YES,expensive,early
22,side_price_time_bucket,311,311,0.854048,0.926045,0.071997,0.071752,0.047505,YES,expensive,middle_9m
14,side_price_time_bucket,146,146,0.503288,0.547945,0.044658,0.067521,0.027485,NO,mid,early
16,side_price_time_bucket,156,156,0.500962,0.544872,0.043910,0.066366,0.026198,NO,mid,middle_9m


## 8. PnL Attribution Summary

This table ties the notebook's findings to the simple decomposition:

`PnL = Selection + Timing + Execution + Sizing - Frictions`

The point is to ask whether the best-looking PnL came from a broad selection effect, timing pocket, execution/friction assumption, capacity, or concentration artifact.


In [8]:
def _fmt_pct(value: Any) -> str:
    return "n/a" if value is None or pd.isna(value) else f"{float(value):.1%}"


def _fmt_money(value: Any) -> str:
    return "n/a" if value is None or pd.isna(value) else f"{float(value):.2f}"

primary_policy_row = _policy_row_or_empty(baseline_results_df, primary_policy)
least_bad_policy_row = baseline_results_df.sort_values(["net_pnl", "net_roi"], ascending=False).iloc[0]
top_policy = primary_policy
top_policy_row = primary_policy_row
top_trades_df = all_trades_df[all_trades_df["policy"].eq(primary_policy)].copy()

largest_day_share = None
largest_market_share = None
if not top_trades_df.empty:
    daily_abs = top_trades_df.groupby("market_close_date")["net_pnl"].sum().abs()
    market_abs = top_trades_df.groupby("market_ticker")["net_pnl"].sum().abs()
    daily_total_abs = float(daily_abs.sum())
    market_total_abs = float(market_abs.sum())
    largest_day_share = float(daily_abs.max() / daily_total_abs) if daily_total_abs else None
    largest_market_share = float(market_abs.max() / market_total_abs) if market_total_abs else None

pnl_attribution_df = pd.DataFrame([
    {
        "row": "Selection",
        "question": "Did the expensive YES selection produce better realized outcomes than implied prices?",
        "finding": f"Primary policy is {primary_policy} with net ROI {_fmt_pct(primary_policy_row['net_roi'])}.",
        "evidence": f"{int(primary_policy_row['trades'])} trades across {int(primary_policy_row['markets'])} markets; net PnL {_fmt_money(primary_policy_row['net_pnl'])}.",
        "severity": "medium" if primary_policy_row["net_pnl"] > 0 else "high",
        "next question": "Does expensive_yes persist across exact 7d and 30d windows?",
    },
    {
        "row": "Timing",
        "question": "Is the result tied to final-window timing or distributed across the full 15-minute window?",
        "finding": "Final-window baselines are visible separately; compare them to expensive_yes before attributing edge to selection.",
        "evidence": baseline_results_df[baseline_results_df["policy"].str.startswith("final_window")][["policy", "trades", "net_roi", "stressed_roi"]].to_dict("records"),
        "severity": "medium",
        "next question": "Does expensive_yes work outside the final three minutes?",
    },
    {
        "row": "Execution / friction",
        "question": "Does the finding survive fees and a simple stressed slippage assumption?",
        "finding": f"Stress ROI is {_fmt_pct(primary_policy_row['stressed_roi'])} under {stress_slippage_assumption:.2f} stressed slippage.",
        "evidence": f"Fee={fee_assumption:.2f}, base slippage={slippage_assumption:.2f}, stress slippage={stress_slippage_assumption:.2f} per contract.",
        "severity": "high" if (pd.isna(primary_policy_row["stressed_roi"]) or primary_policy_row["stressed_roi"] <= 0) else "medium",
        "next question": "What fill slippage and partial-fill behavior appears in private execution or paper logs?",
    },
    {
        "row": "Capacity",
        "question": "Is the apparent opportunity large enough to matter beyond toy sizing?",
        "finding": f"One-contract baseline has {int(primary_policy_row['trades'])} executable rows.",
        "evidence": "This notebook does not prove displayed depth or live fillability; capacity is proxied by row count only.",
        "severity": "medium" if primary_policy_row["trades"] >= 50 else "high",
        "next question": "How much posted depth existed at the selected ask at decision time?",
    },
    {
        "row": "Concentration",
        "question": "Is PnL spread across days and markets, or dominated by one slice?",
        "finding": f"Largest-day share {_fmt_pct(largest_day_share)}; largest-market share {_fmt_pct(largest_market_share)}.",
        "evidence": f"Unique market-close days={int(primary_policy_row['unique_days'])}, unique markets={int(primary_policy_row['markets'])}.",
        "severity": "high" if (largest_day_share is not None and largest_day_share >= 0.50) or primary_policy_row["unique_days"] < 7 else "medium",
        "next question": "Does expensive_yes remain positive when the largest absolute-PnL day is removed?",
    },
])

pnl_attribution_df


,row,question,finding,evidence,severity,next question
0,Selection,Did the expensive YES selection produce better...,Primary policy is expensive_yes with net ROI -...,584 trades across 93 markets; net PnL -22.52.,high,Does expensive_yes persist across exact 7d and...
1,Timing,Is the result tied to final-window timing or d...,Final-window baselines are visible separately;...,"[{'policy': 'final_window_yes', 'trades': 476,...",medium,Does expensive_yes work outside the final thre...
2,Execution / friction,Does the finding survive fees and a simple str...,Stress ROI is -6.3% under 0.02 stressed slippage.,"Fee=0.01, base slippage=0.00, stress slippage=...",high,What fill slippage and partial-fill behavior a...
3,Capacity,Is the apparent opportunity large enough to ma...,One-contract baseline has 584 executable rows.,This notebook does not prove displayed depth o...,medium,How much posted depth existed at the selected ...
4,Concentration,"Is PnL spread across days and markets, or domi...",Largest-day share 100.0%; largest-market share...,"Unique market-close days=1, unique markets=93.",high,Does expensive_yes remain positive when the la...


## 9. Top Finding Drilldown

The primary top policy for this pass is `expensive_yes`. These tables show the current run's daily PnL, leave-largest-day-out robustness, top markets, time-to-expiry breakdown, configurable entry-price sub-buckets, and market/day contribution. The aim is to see whether "expensive YES" is too broad.


In [9]:
def _largest_share(series: pd.Series) -> float | None:
    total = float(series.abs().sum())
    return None if total == 0 else float(series.abs().max() / total)


def _trade_metrics(trades: pd.DataFrame) -> dict[str, float | None]:
    if trades.empty:
        return {"net_pnl": 0.0, "net_roi": None, "stressed_roi": None}
    net_pnl = float(trades["net_pnl"].sum())
    stress_pnl = float(trades["stress_pnl"].sum())
    cost_basis = float(trades["cost_basis"].sum())
    stress_cost_basis = float(trades["stress_cost_basis"].sum())
    return {
        "net_pnl": net_pnl,
        "net_roi": net_pnl / cost_basis if cost_basis else None,
        "stressed_roi": stress_pnl / stress_cost_basis if stress_cost_basis else None,
    }

if top_trades_df.empty:
    daily_pnl_df = pd.DataFrame()
    leave_largest_day_out_df = pd.DataFrame()
    top_markets_df = pd.DataFrame()
    time_to_expiry_distribution_df = pd.DataFrame()
    price_sub_bucket_pnl_df = pd.DataFrame()
    market_day_pnl_df = pd.DataFrame()
    top_finding_summary_df = pd.DataFrame([{"top_policy": primary_policy, "trust_read": "No executable trades were available."}])
else:
    daily_pnl_df = top_trades_df.groupby("market_close_date", as_index=False).agg(
        trades=("market_ticker", "size"),
        markets=("market_ticker", "nunique"),
        net_pnl=("net_pnl", "sum"),
        stressed_pnl=("stress_pnl", "sum"),
    ).sort_values("market_close_date")

    largest_day_row = daily_pnl_df.reindex(daily_pnl_df["net_pnl"].abs().sort_values(ascending=False).index).iloc[0]
    largest_abs_day = largest_day_row["market_close_date"]
    largest_abs_day_pnl = float(largest_day_row["net_pnl"])
    largest_abs_day_share = _largest_share(daily_pnl_df.set_index("market_close_date")["net_pnl"])
    full_metrics = _trade_metrics(top_trades_df)
    ex_largest_day_trades = top_trades_df[~top_trades_df["market_close_date"].eq(largest_abs_day)]
    ex_metrics = _trade_metrics(ex_largest_day_trades)
    leave_largest_day_out_df = pd.DataFrame([
        {
            "metric": "Net PnL",
            "Full sample": full_metrics["net_pnl"],
            "Ex-largest day": ex_metrics["net_pnl"],
            "largest_abs_day": largest_abs_day,
            "largest_abs_day_share": largest_abs_day_share,
            "removed_day_sign": "positive" if largest_abs_day_pnl > 0 else "negative" if largest_abs_day_pnl < 0 else "flat",
        },
        {
            "metric": "Net ROI",
            "Full sample": full_metrics["net_roi"],
            "Ex-largest day": ex_metrics["net_roi"],
            "largest_abs_day": largest_abs_day,
            "largest_abs_day_share": largest_abs_day_share,
            "removed_day_sign": "positive" if largest_abs_day_pnl > 0 else "negative" if largest_abs_day_pnl < 0 else "flat",
        },
        {
            "metric": "Stress ROI",
            "Full sample": full_metrics["stressed_roi"],
            "Ex-largest day": ex_metrics["stressed_roi"],
            "largest_abs_day": largest_abs_day,
            "largest_abs_day_share": largest_abs_day_share,
            "removed_day_sign": "positive" if largest_abs_day_pnl > 0 else "negative" if largest_abs_day_pnl < 0 else "flat",
        },
    ])

    top_markets_df = top_trades_df.groupby("market_ticker", as_index=False).agg(
        trades=("market_ticker", "size"),
        net_pnl=("net_pnl", "sum"),
        stressed_pnl=("stress_pnl", "sum"),
        avg_entry_ask=("side_ask", "mean"),
        market_close_date=("market_close_date", "first"),
        side=("side", "first"),
    ).sort_values("net_pnl", ascending=False).head(12)

    time_to_expiry_distribution_df = top_trades_df.groupby("time_to_expiry_bucket", as_index=False).agg(
        trades=("market_ticker", "size"),
        markets=("market_ticker", "nunique"),
        net_pnl=("net_pnl", "sum"),
        stressed_pnl=("stress_pnl", "sum"),
        avg_seconds_to_expiry=("time_to_expiry_seconds", "mean"),
    ).sort_values("net_pnl", ascending=False)

    price_sub_bucket_pnl_df = top_trades_df.groupby("top_policy_price_sub_bucket", as_index=False).agg(
        trades=("market_ticker", "size"),
        markets=("market_ticker", "nunique"),
        net_pnl=("net_pnl", "sum"),
        stressed_pnl=("stress_pnl", "sum"),
        avg_entry_ask=("side_ask", "mean"),
    ).sort_values("net_pnl", ascending=False)

    market_day_pnl_df = top_trades_df.groupby(["market_close_date", "market_ticker"], as_index=False).agg(
        trades=("market_ticker", "size"),
        net_pnl=("net_pnl", "sum"),
        stressed_pnl=("stress_pnl", "sum"),
        avg_entry_ask=("side_ask", "mean"),
    ).sort_values("net_pnl", ascending=False).head(20)

    day_share = _largest_share(daily_pnl_df.set_index("market_close_date")["net_pnl"])
    market_share = _largest_share(top_trades_df.groupby("market_ticker")["net_pnl"].sum())
    trust_reasons = []
    if primary_policy_row["stressed_roi"] and primary_policy_row["stressed_roi"] > 0:
        trust_reasons.append("survives stressed slippage")
    else:
        trust_reasons.append("does not clearly survive stressed slippage")
    if day_share is not None and day_share >= 0.50:
        trust_reasons.append("largest day dominates PnL")
    if primary_policy_row["unique_days"] < 7:
        trust_reasons.append("too few unique days for persistence evidence")
    if primary_policy_row["markets"] < 30:
        trust_reasons.append("limited market breadth")
    top_finding_summary_df = pd.DataFrame([
        {
            "top_policy": primary_policy,
            "net_pnl": primary_policy_row["net_pnl"],
            "net_roi": primary_policy_row["net_roi"],
            "stressed_roi": primary_policy_row["stressed_roi"],
            "largest_day_share": day_share,
            "largest_market_share": market_share,
            "unique_days": int(primary_policy_row["unique_days"]),
            "unique_markets": int(primary_policy_row["markets"]),
            "why_it_may_be_trustworthy_or_not": "; ".join(trust_reasons),
        }
    ])

for label, table in [
    ("Daily PnL", daily_pnl_df),
    ("Leave-largest-day-out robustness", leave_largest_day_out_df),
    ("Top markets by PnL", top_markets_df),
    ("Time-to-expiry PnL", time_to_expiry_distribution_df),
    ("Entry price sub-bucket PnL", price_sub_bucket_pnl_df),
    ("Market/day PnL", market_day_pnl_df),
    ("Trust read", top_finding_summary_df),
]:
    display(Markdown(f"**{label}**"))
    display(table)


**Daily PnL**

,market_close_date,trades,markets,net_pnl,stressed_pnl
0,2026-06-05,584,93,-22.516,-34.196


**Leave-largest-day-out robustness**

,metric,Full sample,Ex-largest day,largest_abs_day,largest_abs_day_share,removed_day_sign
0,Net PnL,-22.516000,0.0,2026-06-05,1.0,negative
1,Net ROI,-0.042683,NaN,2026-06-05,1.0,negative
2,Stress ROI,-0.063420,NaN,2026-06-05,1.0,negative


**Top markets by PnL**

...

,top_policy_price_sub_bucket,trades,markets,net_pnl,stressed_pnl,avg_entry_ask
1,75_90c,140,43,14.010,11.210,0.825643
0,65_75c,99,47,8.240,6.260,0.694646
2,90_100c,345,93,-44.766,-51.666,0.977728


**Market/day PnL**

,market_close_date,market_ticker,trades,net_pnl,stressed_pnl,avg_entry_ask
39,2026-06-05,KXBTC15M-26JUN050600-00,15,2.761,2.461,0.805933
51,2026-06-05,KXBTC15M-26JUN050915-15,10,2.100,1.900,0.780000
16,2026-06-05,KXBTC15M-26JUN050000-00,16,1.920,1.600,0.870000
15,2026-06-05,KXBTC15M-26JUN042345-45,16,1.907,1.587,0.870812
80,2026-06-05,KXBTC15M-26JUN051630-30,12,1.901,1.661,0.831583
81,2026-06-05,KXBTC15M-26JUN051645-45,15,1.846,1.546,0.866933
69,2026-06-05,KXBTC15M-26JUN051345-45,14,1.762,1.482,0.864143
83,2026-06-05,KXBTC15M-26JUN051715-15,13,1.702,1.442,0.859077
26,2026-06-05,KXBTC15M-26JUN050245-45,13,1.670,1.410,0.861538
89,2026-06-05,KXBTC15M-26JUN051845-45,15,1.630,1.330,0.881333


**Trust read**

,top_policy,net_pnl,net_roi,stressed_roi,largest_day_share,largest_market_share,unique_days,unique_markets,why_it_may_be_trustworthy_or_not
0,expensive_yes,-22.516,-0.042683,-0.06342,1.0,0.049887,1,93,does not clearly survive stressed slippage; la...


## 10. Longer-Window Comparison Plan

- [x] current run: exact `date_end` market-close day
- [x] 7d: exact `date_end - 6 days` through `date_end`
- [x] 30d: exact `date_end - 29 days` through `date_end`
- [ ] 60d: keep as a follow-up once the same cache and completeness rules prove stable

Persistence across windows matters more than one high-ROI short slice. The table below is the compact persistence check: `expensive_yes` is the primary hypothesis, while each window's best policy is only a sanity check.


In [10]:
window_comparison_df


,window,start,end,expected_days,observed_market_close_days,source_status,primary_policy,primary_trades,primary_markets,primary_net_pnl,primary_net_roi,primary_stressed_roi,least_bad_policy,least_bad_policy_positive,least_bad_policy_net_pnl,least_bad_policy_net_roi,least_bad_policy_stressed_roi
0,current,2026-06-05,2026-06-05,1,1,cached_source,expensive_yes,584,93,-22.516,-0.042683,-0.063420,cheap_yes,False,-7.385,-0.085489,-0.203509
1,7d,2026-05-30,2026-06-05,7,7,cached_source,expensive_yes,4180,649,-407.772,-0.108140,-0.127484,cheap_no,False,-38.680,-0.080469,-0.196364
2,30d,2026-05-07,2026-06-05,30,30,cached_source,expensive_yes,18220,2749,-1706.896,-0.104398,-0.123924,cheap_no,False,-95.485,-0.044033,-0.161464


## 11. Candidate Edge Gate

A policy is a candidate structural edge only if it clears positive net ROI, positive stressed ROI, minimum trades, and minimum markets. This makes "no policy passes" a first-class notebook outcome.

In [11]:
candidate_policy_gate_df = baseline_results_df.copy()
candidate_policy_gate_df["positive_net_roi"] = candidate_policy_gate_df["net_roi"].gt(0)
candidate_policy_gate_df["positive_stressed_roi"] = candidate_policy_gate_df["stressed_roi"].gt(0)
candidate_policy_gate_df["enough_trades"] = candidate_policy_gate_df["trades"].ge(minimum_candidate_trades)
candidate_policy_gate_df["enough_markets"] = candidate_policy_gate_df["markets"].ge(minimum_candidate_markets)
candidate_policy_gate_df["candidate_structural_edge"] = candidate_policy_gate_df[
    ["positive_net_roi", "positive_stressed_roi", "enough_trades", "enough_markets"]
].all(axis=1)
candidate_policy_gate_df = candidate_policy_gate_df[
    [
        "policy",
        "trades",
        "markets",
        "net_pnl",
        "net_roi",
        "stressed_roi",
        "positive_net_roi",
        "positive_stressed_roi",
        "enough_trades",
        "enough_markets",
        "candidate_structural_edge",
    ]
]

any_positive_net_roi = bool(candidate_policy_gate_df["positive_net_roi"].any())
any_positive_stressed_roi = bool(candidate_policy_gate_df["positive_stressed_roi"].any())
all_enough_trades = bool(candidate_policy_gate_df["enough_trades"].all())
all_enough_markets = bool(candidate_policy_gate_df["enough_markets"].all())
any_candidate_edge = bool(candidate_policy_gate_df["candidate_structural_edge"].any())

candidate_edge_gate_df = pd.DataFrame([
    {"gate": "Positive net ROI", "result": "passed" if any_positive_net_roi else "failed"},
    {"gate": "Positive stressed ROI", "result": "passed" if any_positive_stressed_roi else "failed"},
    {"gate": f"Enough trades >= {minimum_candidate_trades}", "result": "passed" if all_enough_trades else "failed"},
    {"gate": f"Enough markets >= {minimum_candidate_markets}", "result": "passed" if all_enough_markets else "failed"},
    {"gate": "Candidate structural edge", "result": "yes" if any_candidate_edge else "no"},
])

candidate_edge_summary_sentence = (
    "No tested dumb baseline has positive net ROI in the current exact window."
    if not any_positive_net_roi
    else "At least one tested dumb baseline has positive net ROI in the current exact window."
)

display(candidate_edge_gate_df)
candidate_policy_gate_df


,gate,result
0,Positive net ROI,failed
1,Positive stressed ROI,failed
2,Enough trades >= 100,passed
3,Enough markets >= 10,passed
4,Candidate structural edge,no


,policy,trades,markets,net_pnl,net_roi,stressed_roi,positive_net_roi,positive_stressed_roi,enough_trades,enough_markets,candidate_structural_edge
0,cheap_yes,640,71,-7.385,-0.085489,-0.203509,False,False,True,True,False
1,expensive_yes,584,93,-22.516,-0.042683,-0.063420,False,False,True,True,False
2,cheap_no,482,58,-37.894,-0.566478,-0.621083,False,False,True,True,False
3,final_window_yes,476,96,-45.155,-0.167766,-0.196196,False,False,True,True,False
4,buy_every_yes,1532,96,-50.691,-0.065859,-0.101622,False,False,True,True,False
5,expensive_no,746,95,-53.725,-0.079862,-0.099826,False,False,True,True,False
6,final_window_no,476,96,-57.619,-0.186096,-0.210375,False,False,True,True,False
7,buy_every_no,1532,96,-82.919,-0.092552,-0.122560,False,False,True,True,False


## 12. Hypothesis Summary

The notebook converts top findings into hypotheses. Statuses are intentionally conservative: public-data evidence can support deeper research, but it should not by itself promote a strategy or authorize live deployment.


In [12]:
def _status_for(row: pd.Series) -> str:
    if row["trades"] == 0 or row["net_roi"] is None or pd.isna(row["net_roi"]):
        return "rejected"
    gate_row = candidate_policy_gate_df[candidate_policy_gate_df["policy"].eq(row["policy"])]
    if not gate_row.empty and bool(gate_row.iloc[0]["candidate_structural_edge"]):
        return "deeper_segmentation_candidate"
    return "rejected"

hypothesis_rows = []
primary_window_30d = window_comparison_df[window_comparison_df["window"].eq("30d")].iloc[0]
for _, row in baseline_results_df.head(6).iterrows():
    status = _status_for(row)
    if row["policy"] == primary_policy and primary_window_30d["primary_stressed_roi"] <= 0:
        status = "rejected"
    hypothesis_rows.append({
        "hypothesis statement": f"{row['policy']} may expose a repeatable KXBTC15M market-structure pocket.",
        "evidence summary": f"current trades={int(row['trades'])}, markets={int(row['markets'])}, net ROI={_fmt_pct(row['net_roi'])}, stress ROI={_fmt_pct(row['stressed_roi'])}, unique days={int(row['unique_days'])}",
        "window evidence": "n/a" if row["policy"] != primary_policy else f"30d expensive_yes stress ROI={_fmt_pct(primary_window_30d['primary_stressed_roi'])}, days={int(primary_window_30d['observed_market_close_days'])}",
        "status": status,
        "recommended next action": "continue only through deeper segmentation variables" if status == "rejected" else "inspect the segment before any live or model work",
    })

hypothesis_summary_df = pd.DataFrame(hypothesis_rows)
hypothesis_summary_df


,hypothesis statement,evidence summary,window evidence,status,recommended next action
0,cheap_yes may expose a repeatable KXBTC15M mar...,"current trades=640, markets=71, net ROI=-8.5%,...",n/a,rejected,continue only through deeper segmentation vari...
1,expensive_yes may expose a repeatable KXBTC15M...,"current trades=584, markets=93, net ROI=-4.3%,...","30d expensive_yes stress ROI=-12.4%, days=30",rejected,continue only through deeper segmentation vari...
2,cheap_no may expose a repeatable KXBTC15M mark...,"current trades=482, markets=58, net ROI=-56.6%...",n/a,rejected,continue only through deeper segmentation vari...
3,final_window_yes may expose a repeatable KXBTC...,"current trades=476, markets=96, net ROI=-16.8%...",n/a,rejected,continue only through deeper segmentation vari...
4,buy_every_yes may expose a repeatable KXBTC15M...,"current trades=1532, markets=96, net ROI=-6.6%...",n/a,rejected,continue only through deeper segmentation vari...
5,expensive_no may expose a repeatable KXBTC15M ...,"current trades=746, markets=95, net ROI=-8.0%,...",n/a,rejected,continue only through deeper segmentation vari...


## 13. MVP Verdict

This section is the compact human-read at the end of the notebook. It summarizes what the MVP due-diligence pass supports and what it explicitly does not support yet.


In [13]:
primary_7d = window_comparison_df[window_comparison_df["window"].eq("7d")].iloc[0]
primary_30d = window_comparison_df[window_comparison_df["window"].eq("30d")].iloc[0]
robustness_row = leave_largest_day_out_df[leave_largest_day_out_df["metric"].eq("Stress ROI")].iloc[0] if not leave_largest_day_out_df.empty else None
survives_windows = primary_7d["primary_stressed_roi"] > 0 and primary_30d["primary_stressed_roi"] > 0
survives_leaveout = robustness_row is not None and robustness_row["Ex-largest day"] is not None and robustness_row["Ex-largest day"] > 0

if not any_candidate_edge:
    overall_recommendation = "no_coarse_dumb_baseline_edge"
    rationale = (
        f"{candidate_edge_summary_sentence} "
        f"Reject {primary_policy}: exact 7d stress ROI is {_fmt_pct(primary_7d['primary_stressed_roi'])} "
        f"and exact 30d stress ROI is {_fmt_pct(primary_30d['primary_stressed_roi'])}."
    )
    top_supported_hypothesis = "None; no tested coarse dumb baseline passes the candidate edge gate."
elif survives_windows and survives_leaveout:
    overall_recommendation = "deeper_research"
    rationale = f"{primary_policy} survives current, 7d, 30d, stressed slippage, and current leave-largest-day-out checks, but public candlesticks still do not prove live fillability."
    top_supported_hypothesis = hypothesis_summary_df[hypothesis_summary_df["status"].ne("rejected")].iloc[0]["hypothesis statement"]
elif survives_windows:
    overall_recommendation = "deeper_research"
    rationale = f"{primary_policy} survives exact 7d and 30d stressed-window checks, but current leave-largest-day-out or concentration remains a concern."
    top_supported_hypothesis = hypothesis_summary_df[hypothesis_summary_df["status"].ne("rejected")].iloc[0]["hypothesis statement"]
else:
    overall_recommendation = "inconclusive"
    rationale = f"{primary_policy} does not consistently survive the required exact-window stressed checks."
    top_supported_hypothesis = "None; required exact-window checks did not support a candidate."

final_verdict_sentence = (
    f"No coarse dumb-baseline structural edge found. Reject {primary_policy}. "
    "Continue with deeper segmentation, not live deployment or model training."
)

mvp_verdict_df = pd.DataFrame([
    {
        "overall recommendation": overall_recommendation,
        "rationale": rationale,
        "top supported hypothesis": top_supported_hypothesis,
        "top risk": "Public candlesticks and one-contract baselines do not prove live fillability, and coarse price/side buckets may be too broad.",
        "recommended next experiment": "Continue with deeper segmentation: threshold distance, quote age, time-to-expiry sub-buckets, spread, settlement-window state, and volatility regime.",
        "what not to do yet": "Do not train a model, deploy live trading, wire database writes, modify runtime code, or treat this notebook as a launch gate.",
        "final verdict": final_verdict_sentence,
    }
])

mvp_verdict_df


,overall recommendation,rationale,top supported hypothesis,top risk,recommended next experiment,what not to do yet,final verdict
0,no_coarse_dumb_baseline_edge,No tested dumb baseline has positive net ROI i...,None; no tested coarse dumb baseline passes th...,Public candlesticks and one-contract baselines...,Continue with deeper segmentation: threshold d...,"Do not train a model, deploy live trading, wir...",No coarse dumb-baseline structural edge found....


## 14. Artifact Export

The notebook exports a small set of durable, human-readable artifacts under `output_dir`. These are research outputs only; they do not mutate a database or control any live trading surface.


In [14]:
def _json_default(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (datetime, date)):
        return value.isoformat()
    if pd.isna(value) if not isinstance(value, (list, dict, tuple, set)) else False:
        return None
    return value


def _write_csv(frame: pd.DataFrame, path: Path) -> None:
    serializable = frame.copy()
    for column in serializable.columns:
        serializable[column] = serializable[column].map(
            lambda value: json.dumps(value, default=_json_default)
            if isinstance(value, (list, dict, tuple))
            else value
        )
    serializable.to_csv(path, index=False)


def _markdown_table(frame: pd.DataFrame, max_rows: int = 12) -> str:
    if frame.empty:
        return "_No rows._"
    preview = frame.head(max_rows).copy().fillna("")
    columns = [str(column) for column in preview.columns]
    lines = ["| " + " | ".join(columns) + " |", "| " + " | ".join(["---"] * len(columns)) + " |"]
    for _, row in preview.iterrows():
        values = []
        for column in preview.columns:
            value = row[column]
            if isinstance(value, (list, dict, tuple)):
                text = json.dumps(value, default=_json_default)
            else:
                text = str(value)
            values.append(text.replace("\n", " ").replace("|", "\\|"))
        lines.append("| " + " | ".join(values) + " |")
    return "\n".join(lines)

output_dir.mkdir(parents=True, exist_ok=True)

top_opportunities_df = baseline_results_df.copy()
top_opportunities_df["largest_day_share"] = None
top_opportunities_df["largest_market_share"] = None
for index, row in top_opportunities_df.iterrows():
    trades = all_trades_df[all_trades_df["policy"].eq(row["policy"])]
    if trades.empty:
        continue
    top_opportunities_df.at[index, "largest_day_share"] = _largest_share(trades.groupby("market_close_date")["net_pnl"].sum())
    top_opportunities_df.at[index, "largest_market_share"] = _largest_share(trades.groupby("market_ticker")["net_pnl"].sum())
top_opportunities_df = top_opportunities_df.sort_values(["net_pnl", "net_roi"], ascending=False)

artifact_paths = {
    "manifest": output_dir / "manifest.json",
    "report": output_dir / "report.md",
    "data_quality_audit": output_dir / "data_quality_audit.csv",
    "baseline_results": output_dir / "baseline_results.csv",
    "calibration_results": output_dir / "calibration_results.csv",
    "pnl_attribution": output_dir / "pnl_attribution.csv",
    "top_opportunities": output_dir / "top_opportunities.csv",
    "window_comparison": output_dir / "window_comparison.csv",
    "candidate_edge_gate": output_dir / "candidate_edge_gate.csv",
    "candidate_policy_gate": output_dir / "candidate_policy_gate.csv",
}

_write_csv(data_quality_audit_df, artifact_paths["data_quality_audit"])
_write_csv(baseline_results_df, artifact_paths["baseline_results"])
_write_csv(calibration_results_df, artifact_paths["calibration_results"])
_write_csv(pnl_attribution_df, artifact_paths["pnl_attribution"])
_write_csv(top_opportunities_df, artifact_paths["top_opportunities"])
_write_csv(window_comparison_df, artifact_paths["window_comparison"])
_write_csv(candidate_edge_gate_df, artifact_paths["candidate_edge_gate"])
_write_csv(candidate_policy_gate_df, artifact_paths["candidate_policy_gate"])

manifest = {
    "schema_version": "kxbtc15m_market_structure_due_diligence_mvp.v3",
    "created_at": _iso(datetime.now(UTC)),
    "market_series": market_series,
    "date_start": date_start.isoformat(),
    "date_end": date_end.isoformat(),
    "source_mode": source_mode,
    "source_provenance": source_provenance,
    "window_source_summary": window_source_summary_df.to_dict("records"),
    "assumptions": assumption_rows,
    "price_buckets": list(price_buckets),
    "time_to_expiry_buckets": list(time_to_expiry_buckets),
    "top_policy_price_sub_buckets": list(top_policy_price_sub_buckets),
    "candidate_gate_thresholds": {
        "minimum_candidate_trades": int(minimum_candidate_trades),
        "minimum_candidate_markets": int(minimum_candidate_markets),
        "requires_positive_net_roi": True,
        "requires_positive_stressed_roi": True,
    },
    "candidate_edge_gate": candidate_edge_gate_df.to_dict("records"),
    "row_counts": {
        "normalized_rows": int(len(rows_df)),
        "markets": int(rows_df["market_ticker"].nunique()),
        "trades_in_all_baselines": int(len(all_trades_df)),
    },
    "primary_policy": primary_policy,
    "least_bad_current_policy": least_bad_policy_row["policy"],
    "least_bad_current_policy_positive": bool(least_bad_policy_row["net_roi"] > 0 and least_bad_policy_row["stressed_roi"] > 0),
    "overall_recommendation": overall_recommendation,
    "final_verdict": final_verdict_sentence,
    "artifact_paths": {key: str(path) for key, path in artifact_paths.items()},
}
artifact_paths["manifest"].write_text(json.dumps(manifest, indent=2, default=_json_default) + "\n", encoding="utf-8")

report_lines = [
    f"# {market_series} Market Structure Due Diligence MVP Report",
    "",
    "This report is generated by the lightweight notebook workbench. It is not a trading bot, model-training artifact, live-deployment gate, runtime change, database writer, or agent orchestration surface.",
    "",
    "## Parameters",
    _markdown_table(pd.DataFrame(assumption_rows)),
    "",
    "## Exported Artifacts",
    _markdown_table(pd.DataFrame([
        {"artifact": key, "file": path.name, "path": str(path)}
        for key, path in artifact_paths.items()
    ])),
    "",
    "## Exact Window Source Summary",
    _markdown_table(window_source_summary_df),
    "",
    "## Window Comparison",
    _markdown_table(window_comparison_df),
    "",
    "## Candidate Edge Gate",
    candidate_edge_summary_sentence,
    "",
    _markdown_table(candidate_edge_gate_df),
    "",
    "### Policy Detail",
    _markdown_table(candidate_policy_gate_df),
    "",
    "## Data Quality Audit",
    _markdown_table(data_quality_audit_df),
    "",
    "## Dumb Baselines",
    _markdown_table(baseline_results_df),
    "",
    "## Calibration Preview",
    _markdown_table(calibration_results_df),
    "",
    "## PnL Attribution",
    _markdown_table(pnl_attribution_df),
    "",
    "## Leave-Largest-Day-Out",
    _markdown_table(leave_largest_day_out_df),
    "",
    "## Expensive YES Breakdown",
    "",
    "### Time-to-expiry",
    _markdown_table(time_to_expiry_distribution_df),
    "",
    "### Entry price sub-bucket",
    _markdown_table(price_sub_bucket_pnl_df),
    "",
    "### Market/day",
    _markdown_table(market_day_pnl_df),
    "",
    "## Top Finding",
    _markdown_table(top_finding_summary_df),
    "",
    "## Hypotheses",
    _markdown_table(hypothesis_summary_df),
    "",
    "## MVP Verdict",
    _markdown_table(mvp_verdict_df),
    "",
]
artifact_paths["report"].write_text("\n".join(report_lines), encoding="utf-8")

pd.DataFrame([{"artifact": key, "path": str(path)} for key, path in artifact_paths.items()])

,artifact,path
0,manifest,/Users/sid/projects/alphadb/artifacts/market-d...
1,report,/Users/sid/projects/alphadb/artifacts/market-d...
2,data_quality_audit,/Users/sid/projects/alphadb/artifacts/market-d...
3,baseline_results,/Users/sid/projects/alphadb/artifacts/market-d...
4,calibration_results,/Users/sid/projects/alphadb/artifacts/market-d...
5,pnl_attribution,/Users/sid/projects/alphadb/artifacts/market-d...
6,top_opportunities,/Users/sid/projects/alphadb/artifacts/market-d...
7,window_comparison,/Users/sid/projects/alphadb/artifacts/market-d...
8,candidate_edge_gate,/Users/sid/projects/alphadb/artifacts/market-d...
9,candidate_policy_gate,/Users/sid/projects/alphadb/artifacts/market-d...
